# 提交退步診斷：不重訓受損分類器
只做 CSV 比對、X-only 量測診斷與既有預測的 BBSE。需要網路與 repository 存取權限；本 notebook 尚未在 Colab 平台執行。若 GitHub repository 私有，先用自己的安全方式在 Colab 授權，勿把 token 寫進 notebook。

In [ ]:
from pathlib import Path
import subprocess,sys,os,tempfile
repo=Path("/content/text")
if not repo.exists():
    subprocess.run(["git","clone","--branch","research-0680-20261024","https://github.com/yeyu0302/text.git",str(repo)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-r",str(repo/"research_0680/configs/requirements.txt")],check=True)
os.chdir(repo)


In [ ]:
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
# 每次建立新目錄，不覆蓋既有結果。
output=Path(tempfile.mkdtemp(prefix="phase6-audit-"))
env["RESEARCH0680_PHASE6_RESULTS"]=str(output)
subprocess.run([sys.executable,"-m","research_0680.phase6.run_all"],env=env,check=True)
print("New output:",output)


In [ ]:
import pandas as pd
display(pd.read_csv(output/"wiener_summary.csv"))
display(pd.read_csv(output/"submission_summary.csv"))
display(pd.read_csv(output/"bbse_prior_diagnostic.csv"))


In [ ]:
subprocess.run([sys.executable,"-m","pytest","-q","research_0680/phase6/tests"],env=env,check=True)
from google.colab import files
files.download(str(repo/"research_0680/phase6/downloads/submission_recovery_known_best_0679.csv"))
